Set up workspace and Hail.

In [1]:
import os
bucket = os.getenv("WORKSPACE_BUCKET")
bucket

'gs://fc-secure-900e2ab3-7090-4920-92b3-ab58dd918fdd'

In [2]:
import hail as hl
hl.default_reference(new_default_reference = "GRCh38")

Loading BokehJS ...

Initializing Hail with default parameters...
/opt/conda/lib/python3.10/site-packages/hailtop/aiocloud/aiogoogle/user_config.py:43: UserWarning:

Reading spark-defaults.conf to determine GCS requester pays configuration. This is deprecated. Please use `hailctl config set gcs_requester_pays/project` and `hailctl config set gcs_requester_pays/buckets`.

Running on Apache Spark version 3.5.3
SparkUI available at http://all-of-us-18830-m.us-central1-c.c.terra-vpc-sc-67e3fdda.internal:45959
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.134-952ae203dbbe
LOGGING: writing to /home/jupyter/workspaces/geneclustersimplicatedincardiovasculardiseaseandmentalhealth/hail-20250701-1629-0.2.134-952ae203dbbe.log


In [3]:
hl.spark_context()

<SparkContext master=yarn appName=pyspark-shell>

Read VDS

In [4]:
vds_srwgs_path = os.getenv("WGS_VDS_PATH")
vds_srwgs_path

'gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/vds/hail.vds'

In [5]:
vds = hl.vds.read_vds(vds_srwgs_path)

Extract region around FADS

In [6]:
fads_interval = ['chr11:61000000-63000000']

In [7]:
fads = hl.vds.filter_intervals(
    vds,
    [hl.parse_locus_interval(x,)
     for x in fads_interval])
fads = hl.vds.filter_chromosomes(fads, keep= ["chr11"])

In [8]:
fads.variant_data.count()

(867057, 414830)

Convert to dense MatrixTable (following the `03_Manipulate Hail VariantDataset` notebook from the "How to Work with All of Us Genomic Data (Hail - Plink)(v8)" example workspace)

In [9]:
mt = fads.variant_data.annotate_entries(AD = hl.vds.local_to_global(fads.variant_data.LAD, 
                                                                    fads.variant_data.LA, 
                                                                    n_alleles=hl.len(fads.variant_data.alleles), 
                                                                    fill_value=0, number='R'))

In [10]:
mt = mt.annotate_entries(GT = hl.vds.lgt_to_gt(mt.LGT, mt.LA))

In [11]:
mt = mt.transmute_entries(FT = hl.if_else(mt.FT, "PASS", "FAIL"))

In [12]:
mt = hl.vds.to_dense_mt(hl.vds.VariantDataset(fads.reference_data, mt))

In [13]:
mt = mt.annotate_rows(info = hl.agg.call_stats(mt.GT, mt.alleles))

In [14]:
fields_to_drop_list = ['as_vets','as_vqsr','LAD', 'LGT', 'LA',
            'tranche_data', 'truth_sensitivity_snp_threshold', 
             'truth_sensitivity_indel_threshold','snp_vqslod_threshold','indel_vqslod_threshold']

In [15]:
mt = mt.drop(*(f for f in fields_to_drop_list if f in mt.entry or f in mt.row or f in mt.col or f in mt.globals))

In [16]:
mt.describe()

----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
----------------------------------------
Row fields:
    'locus': locus<GRCh38>
    'alleles': array<str>
    'filters': set<str>
    'info': struct {
        AC: array<int32>, 
        AF: array<float64>, 
        AN: int32, 
        homozygote_count: array<int32>
    }
----------------------------------------
Entry fields:
    'GT': call
    'GQ': int32
    'PS': int64
    'FT': str
    'AD': array<int32>
    'RGQ': int32
----------------------------------------
Column key: ['s']
Row key: ['locus', 'alleles']
----------------------------------------


In [17]:
mt.count()

(867057, 414830)

In [18]:
out_path = f'{bucket}/data/srWGS/FADS/chr11_61000000-63000000.mt'
mt.write(out_path, overwrite = True)

Convert to VCF

In [19]:
mt_vcf = mt.transmute_rows(info = mt.info.annotate(AF=mt.info.AF[1:], 
                                                   AC=mt.info.AC[1:]))
vcf_header = "gs://fc-secure-ff68a895-e88d-426e-9ae4-b3802c51b53b/data/vcf_header.txt"
metadata = hl.get_vcf_metadata(vcf_header)

In [20]:
out_vcf = f'{bucket}/data/srWGS/FADS/chr11_61000000-63000000.vcf.bgz'
hl.export_vcf(mt_vcf, out_vcf, tabix = False, metadata=metadata)

In [21]:
hl.stop()

Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x7f92fd87f820>
Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x7f92fd87c310>
